In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('covid_toy.csv')

In [3]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [4]:
# here, there are different types of columns, 
# - age, fever: already numerical 
# - gender, city: nominal 
# - cough: ordinal 
# so one way is to transform each feature separately like what 
# we have have done so far. Or there is a smart way - ColumnTransformer

EDA

In [5]:
df.isna().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

fever column has missing values, so Imputation feature transform could be applied here, age is fine. So either we can apply standardization or leave it

In [8]:
df['cough'].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

Data prep

In [12]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['has_covid']), df['has_covid'], test_size=0.2, random_state=42)

In [13]:
X_train

,age,gender,fever,cough,city
55,81,Female,101.0,Mild,Mumbai
88,5,Female,100.0,Mild,Kolkata
26,19,Female,100.0,Mild,Kolkata
42,27,Male,100.0,Mild,Delhi
69,73,Female,103.0,Mild,Delhi
...,...,...,...,...,...
60,24,Female,102.0,Strong,Bangalore
71,75,Female,104.0,Strong,Delhi
14,51,Male,104.0,Mild,Bangalore
92,82,Female,102.0,Strong,Kolkata


### 1. Aam Zindagi

In [67]:
# applying simple imputer to fever column
'''
SimpleImputer() is a scikit learn transformer that replaces missing
values in a dataset using specified univariate strategies per column.
It operates by first fitting on training data to calculate statistics
eg., mean, median, mode and then transforming data to fill missing entries,
ensuring consistent imputation across train/test splits.
strategy = 'mean' or 'median' for numeric data,
strategy = 'most_frequent' for categorical data, or
strategy = 'constant' to fill with user-defined fill_value.
'''
from sklearn.impute import SimpleImputer
si = SimpleImputer()    # add_indicator=True to highlight which was the missing value in original data
X_train_fever = si.fit_transform(X_train[['fever']])

X_test_fever = si.fit_transform(X_test[['fever']])

In [68]:
X_train_fever.shape

(80, 1)

In [69]:
# OrdinalEncoding -> 'cough'
from sklearn.preprocessing import OrdinalEncoder
oe = OrdinalEncoder(categories=[['Mild', 'Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

X_test_cough = oe.transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [70]:
# One Hot encoding -> gender, city
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(drop='first', sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender', 'city']])

X_test_gender_city = ohe.transform(X_test[['gender', 'city']])
X_train_gender_city.shape

(80, 4)

In [ ]:
# Extracting age
X_train_age = X_train.drop(columns=['gender', 'fever', 'cough', 'city']).values
X_train_age #--- .values -> gives a numpy array
X_test_age = X_test.drop(columns=['gender', 'fever', 'cough', 'city']).values
X_train_age.shape

(80, 1)

In [72]:
# combining everything
X_train_transformed = np.concatenate((X_train_age, X_train_fever, X_train_gender_city, X_train_cough), axis=1)
X_test_transformed = np.concatenate((X_test_age, X_test_fever, X_test_gender_city, X_test_cough), axis=1)
X_train_transformed.shape

(80, 7)

In [73]:
X_train_transformed

array([[ 81., 101.,   0.,   0.,   0.,   1.,   0.],
       [  5., 100.,   0.,   0.,   1.,   0.,   0.],
       [ 19., 100.,   0.,   0.,   1.,   0.,   0.],
       [ 27., 100.,   1.,   1.,   0.,   0.,   0.],
       [ 73., 103.,   0.,   1.,   0.,   0.,   0.],
       [ 70., 103.,   1.,   0.,   1.,   0.,   1.],
       [ 49., 102.,   0.,   1.,   0.,   0.,   0.],
       [ 51., 101.,   0.,   0.,   1.,   0.,   1.],
       [ 64., 101.,   0.,   1.,   0.,   0.,   0.],
       [ 83., 101.,   0.,   0.,   1.,   0.,   0.],
       [ 65.,  98.,   0.,   0.,   0.,   1.,   0.],
       [ 18., 104.,   0.,   0.,   0.,   0.,   0.],
       [ 16., 103.,   0.,   0.,   0.,   0.,   0.],
       [ 16., 104.,   1.,   0.,   1.,   0.,   0.],
       [ 27., 100.,   1.,   0.,   1.,   0.,   0.],
       [ 84., 101.,   0.,   0.,   0.,   0.,   0.],
       [ 51., 104.,   1.,   0.,   1.,   0.,   0.],
       [ 69., 102.,   0.,   0.,   0.,   0.,   0.],
       [ 82., 102.,   0.,   0.,   0.,   0.,   1.],
       [ 69., 103.,   0.,   0.,

### Mentos Zindagi

In [74]:
from sklearn.compose import ColumnTransformer

In [75]:
transformer = ColumnTransformer(transformers=[
    ('txf1', SimpleImputer(), ['fever']),
    ('txf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
    ('txf3', OneHotEncoder(sparse_output=False, drop='first'), ['gender','city'])
], remainder='passthrough')

In [76]:
transformer.fit_transform(X_train).shape

(80, 7)

In [77]:
transformer.transform(X_test).shape

(20, 7)